# Four-stock SDE-GAN: data inspection and benchmark comparison

Inspect the four-stock dataset prepared from `Data/eligible/price_histories.csv` and compare:
- **Unconditional models:** Fixed `HistoricalSimulation` and the stabilized, mean-anchored `SDE-GAN` (per-seed + across-seed spread).
- **Conditional rolling models:** Rolling 252-session `HistoricalSimulation` and zero-shot `TSFMForecastBenchmark` (`thuml/sundial-base-128m`).

Before running this notebook, run from the repository root:
```sh
uv run python experiments/4stocks/scripts/prepare_data.py
uv run python experiments/4stocks/scripts/train.py 0 1 2
uv run --extra tsfm python experiments/4stocks/scripts/run_tsfm.py
```

In [ ]:
from pathlib import Path
import json
import os
import sys

os.environ.setdefault('JAX_PLATFORMS', 'cpu')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'benchmarks/historical.py').is_file())
EXP_DIR = ROOT / 'experiments/4stocks'
DATA_DIR = EXP_DIR / 'data'
RUNS_DIR = EXP_DIR / 'results/sde_gan'
for p in (ROOT, EXP_DIR / 'model', EXP_DIR / 'scripts'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from benchmarks.historical import HistoricalSimulation, HistoricalSimulationConfig
from evaluate import (
    CONFIDENCE_LEVELS, ROLLING_LOOKBACK_SESSIONS, SUNDIAL_RELEASE_DATE,
    acf, load_observed, load_tsfm_forecasts, risk_rows, score_backtest_frame, terminal_returns,
)
from sde_gan import SDEGAN

SEEDS = (0, 1, 2)
N_PATHS = 10_000
SAMPLE_SEED = 100
CHECKPOINT = 'selected'  # 'selected' (lowest val MMD) or e.g. 'step_2000'
ROLLING_LOOKBACK = ROLLING_LOOKBACK_SESSIONS
EVALUATE_TEST = False    # Keep False until the baseline protocol is frozen.

meta = json.loads((DATA_DIR / 'preprocessing.json').read_text())
STOCKS = meta['stock_order']
MU = np.asarray(meta['training_mean'], dtype=np.float64)
STD = np.asarray(meta['training_std'], dtype=np.float64)
dates = pd.read_csv(DATA_DIR / 'window_dates.csv', parse_dates=['start_date', 'end_date'])

print('Source:', meta['source_file'])
print('Stocks:', STOCKS)
pd.DataFrame(meta['splits']).T

## 1. Observed splits and historical simulation benchmarks (fixed & rolling)

In [ ]:
eval_splits = ('validation', 'test') if EVALUATE_TEST else ('validation',)
frames, observed = {}, {}
for split in ('train', *eval_splits):
    frames[split], observed[split] = load_observed(split, STOCKS, dates)

hist_common = dict(
    assets=tuple(STOCKS),
    weights={s: 1.0 / len(STOCKS) for s in STOCKS},
    return_type='simple',
    horizon_sessions=meta['window_length'],
    rebalance='daily',
    confidence_levels=CONFIDENCE_LEVELS,
    min_scenarios=100,
)
historical_fixed = HistoricalSimulation(
    HistoricalSimulationConfig(**hist_common, estimation='fixed')
).fit(frames['train'])
hist_summary = historical_fixed.summarize()
np.testing.assert_allclose(hist_summary.scenarios.portfolio_return,
                           terminal_returns(observed['train']), rtol=0, atol=1e-14)

rolling_forecasts = {}
for split in eval_splits:
    causal_history = (frames['train'] if split == 'validation'
                      else pd.concat([frames['train'], frames['validation']], ignore_index=True))
    rolling_forecasts[split] = HistoricalSimulation(HistoricalSimulationConfig(
        **hist_common, estimation='rolling', lookback_sessions=ROLLING_LOOKBACK,
    )).fit(causal_history).backtest(frames[split]).forecasts

split_stats = []
for split in ('train', *eval_splits):
    port = terminal_returns(observed[split])
    loss = -port
    split_stats.append({
        'split': split, 'windows': len(port),
        'port_mean_pct': 100 * port.mean(), 'port_std_pct': 100 * port.std(ddof=1),
        'loss_q95_pct': 100 * np.quantile(loss, 0.95), 'loss_q99_pct': 100 * np.quantile(loss, 0.99),
    })
display(pd.DataFrame(split_stats).set_index('split').round(3))
display(hist_summary.risk)

## 2. Load saved SDE-GAN seeds and TSFM forecasts

In [ ]:
generated, histories, inventory = {}, {}, []
for seed in SEEDS:
    run_dir = RUNS_DIR / f'seed_{seed}'
    if not (run_dir / 'completion.json').is_file():
        print(f'Skipping missing SDE-GAN seed {seed} ({run_dir})')
        continue
    load_dir = run_dir if CHECKPOINT == 'selected' else run_dir / 'checkpoints' / CHECKPOINT
    assert json.loads((load_dir / 'preprocessing.json').read_text()) == meta
    model = SDEGAN.load(load_dir)
    standardized = model.sample(N_PATHS, SAMPLE_SEED)
    generated[seed] = np.expm1(standardized.astype(np.float64) * STD + MU)
    histories[seed] = pd.read_csv(run_dir / 'history.csv')
    inventory.append({
        'seed': seed, 'selected_step': model.run_metadata['selected_step'],
        'validation_mmd2': model.run_metadata['validation_score'],
        'train_mean_error': model.run_metadata.get('train_mean_error'),
    })

tsfm_forecasts = {}
for split in eval_splits:
    loaded = load_tsfm_forecasts(split, meta)
    if loaded is not None:
        tsfm_forecasts[split] = loaded
        print(f'Loaded TSFM {split} forecasts: {len(loaded)} rows')
    else:
        print(f'No saved TSFM {split} forecasts yet. Run: uv run --extra tsfm python experiments/4stocks/scripts/run_tsfm.py')

if inventory:
    display(pd.DataFrame(inventory).set_index('seed').round(6))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for seed, h in histories.items():
        val = h.dropna(subset=['validation_mmd2'])
        axes[0].plot(val['step'], val['validation_mmd2'], marker='.', label=f'seed {seed}')
        if 'train_mean_error' in val:
            axes[1].plot(val['step'], val['train_mean_error'], marker='.', label=f'seed {seed}')
    axes[0].set(title='Validation multi-scale MMD²', xlabel='Generator step', ylabel='MMD²')
    axes[1].set(title='Training mean RMSE (std units)', xlabel='Generator step', ylabel='RMSE')
    for ax in axes:
        ax.legend()
    plt.show()
else:
    print('No completed SDE-GAN runs found yet. Run: uv run python experiments/4stocks/scripts/train.py 0 1 2')

## 3. Unified portfolio risk comparison: Fixed HS, Rolling HS, TSFM (Sundial), and SDE-GAN

**Methodological notes:**
- **Unconditional vs. conditional:** Fixed HS and SDE-GAN output one fixed VaR/ES distribution; Rolling HS (252d) and TSFM (Sundial, 252d context) adapt their VaR/ES at every forecast origin.
- **Sundial pretraining cutoff:** `thuml/sundial-base-128m` was released in **February 2025** (`arXiv:2502.00816`). Validation (`2023-09` .. `2025-03`) is mostly pre-release, whereas the reserved test split (`2025-03` .. `2026-09`) is strictly post-release.

In [ ]:
for split in eval_splits:
    realized = -terminal_returns(observed[split])
    rows = [{**r, 'model': 'historical_fixed', 'seed': -1} for r in risk_rows(observed['train'], realized)]
    rows.extend({**r, 'model': f'historical_rolling_{ROLLING_LOOKBACK}', 'seed': -1}
                for r in score_backtest_frame(rolling_forecasts[split]))
    if split in tsfm_forecasts:
        rows.extend({**r, 'model': 'tsfm_sundial', 'seed': -1}
                    for r in score_backtest_frame(tsfm_forecasts[split]))
    for seed, paths in generated.items():
        rows.extend({**r, 'model': 'sde_gan', 'seed': seed} for r in risk_rows(paths, realized))
    df = pd.DataFrame(rows)
    print(f'=== {split.upper()} RISK SCORES ===')
    display(df.set_index(['model', 'seed', 'confidence']).round(4))

    # Bar chart of quantile loss across all available models
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for confidence, ax in zip(CONFIDENCE_LEVELS, axes):
        sub = df.loc[df.confidence.eq(confidence)]
        bench = sub.loc[~sub.model.eq('sde_gan')]
        gan = sub.loc[sub.model.eq('sde_gan')]
        labels = list(bench.model) + [f'sde_gan s{s}' for s in gan.seed]
        vals = list(bench.quantile_loss_pct) + list(gan.quantile_loss_pct)
        colors = ['C1'] * len(bench) + ['C0'] * len(gan)
        ax.bar(labels, vals, color=colors)
        ax.set(title=f'{confidence:.0%} {split} quantile loss (lower is better)', ylabel='Quantile loss × 100')
        ax.tick_params(axis='x', rotation=25)
    plt.show()

    # Time-series plot: realized 20-session loss vs time-varying and fixed VaR forecasts
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
    r_bt = rolling_forecasts[split]
    for confidence, ax in zip(CONFIDENCE_LEVELS, axes):
        rsub = r_bt.loc[r_bt.confidence.eq(confidence)].sort_values('target_start')
        t = rsub['target_start']
        ax.plot(t, 100.0 * rsub['realized_loss'], color='black', alpha=0.45, linewidth=1.0, label='Realized 20d loss')
        fixed_var = df.loc[df.model.eq('historical_fixed') & df.confidence.eq(confidence), 'var_pct'].iloc[0]
        ax.axhline(fixed_var, color='grey', linestyle='--', linewidth=1.3, label=f'Fixed HS ({fixed_var:.2f}%)')
        ax.plot(t, 100.0 * rsub['var'], color='C2', linewidth=1.5, label=f'Rolling HS {ROLLING_LOOKBACK}d')
        if generated:
            gvars = df.loc[df.model.eq('sde_gan') & df.confidence.eq(confidence), 'var_pct']
            ax.axhline(gvars.mean(), color='C0', linewidth=1.6, label=f'SDE-GAN mean ({gvars.mean():.2f}%)')
            ax.axhspan(gvars.min(), gvars.max(), color='C0', alpha=0.12)
        if split in tsfm_forecasts:
            tsub = tsfm_forecasts[split].loc[tsfm_forecasts[split].confidence.eq(confidence)].sort_values('target_start')
            ax.plot(tsub['target_start'], 100.0 * tsub['var'], color='C3', linewidth=1.6, label='TSFM (Sundial)')
            breach_pts = tsub.loc[tsub['breach'].fillna(False)]
            ax.scatter(breach_pts['target_start'], 100.0 * breach_pts['realized_loss'],
                       color='C3', s=16, zorder=5, label='TSFM breach')
        if t.min() <= SUNDIAL_RELEASE_DATE <= t.max():
            ax.axvline(SUNDIAL_RELEASE_DATE, color='C3', linestyle=':', alpha=0.7, label='Sundial release (Feb 2025)')
        ax.set(title=f'{confidence:.0%} VaR vs. realized 20-session portfolio loss ({split})', ylabel='20-session loss (%)')
        ax.legend(fontsize=8, ncol=3, loc='upper right')
    axes[-1].set_xlabel('Target window start date')
    plt.show()

## 4. Multivariate distribution faithfulness (Observed vs. SDE-GAN)

TSFM forecasts the 1D portfolio series directly, so cross-stock correlation and per-stock ACF diagnostics apply to the multivariate SDE-GAN versus observed data.

In [ ]:
if generated:
    train_flat = observed['train'].reshape(-1, len(STOCKS))
    train_mean, train_std = train_flat.mean(0), train_flat.std(0, ddof=1)
    train_corr = np.corrcoef(train_flat.T)
    train_signed, train_abs = acf(observed['train']), acf(observed['train'], True)
    pairs = np.triu_indices(len(STOCKS), 1)

    dist_rows = []
    for seed, paths in generated.items():
        flat = paths.reshape(-1, len(STOCKS))
        port = terminal_returns(paths)
        dist_rows.append({
            'seed': seed,
            'port_mean_pct': 100 * port.mean(),
            'port_std_pct': 100 * port.std(ddof=1),
            'mean_rmse_std': np.sqrt(np.mean(((flat.mean(0) - train_mean) / train_std) ** 2)),
            'vol_rmse_train': np.sqrt(np.mean((flat.std(0, ddof=1) / train_std - 1) ** 2)),
            'corr_rmse_train': np.sqrt(np.mean((np.corrcoef(flat.T) - train_corr)[pairs] ** 2)),
            'signed_acf_rmse': np.sqrt(np.mean((acf(paths) - train_signed) ** 2)),
            'abs_acf_rmse': np.sqrt(np.mean((acf(paths, True) - train_abs) ** 2)),
        })
    display(pd.DataFrame(dist_rows).set_index('seed').round(4))

    fig, axes = plt.subplots(2, len(STOCKS), figsize=(15, 6), constrained_layout=True)
    for row_idx, (label, is_abs) in enumerate((('Signed', False), ('Absolute', True))):
        for j, stock in enumerate(STOCKS):
            ax = axes[row_idx, j]
            ax.plot(range(1, 6), acf(observed['train'], is_abs)[:, j], color='black', label='train')
            ax.plot(range(1, 6), acf(observed['validation'], is_abs)[:, j], color='grey', label='validation')
            curves = np.stack([acf(paths, is_abs)[:, j] for paths in generated.values()])
            ax.plot(range(1, 6), curves.mean(0), color='C0', label='SDE-GAN mean')
            ax.fill_between(range(1, 6), curves.min(0), curves.max(0), color='C0', alpha=0.18)
            ax.set(title=f'{stock}: {label}', xlabel='Lag', ylabel='ACF', xticks=range(1, 6))
            ax.legend(fontsize=7)
    plt.show()